# TED search-field hierarchy

Read the **first column** of `List_of_search_fields.csv` and recursively group its hyphen-separated names. Only Python's standard library is needed.

1. Group distinct names by their first word, retaining distinct remaining suffixes. A name ending at the current key contributes `None`.
2. Move common leading words from **all** suffixes into their key, stopping at a branch or a terminal suffix. Empty suffixes become `None`.
3. Repeat for each suffix group until every leaf is `[None]`, returning a list of single-key maps sorted by their compressed keys at every level.

`None` is Python's terminal marker, not the string `"none"`. When a name is also a prefix of longer names, a bare `None` list element preserves that name. For example, `['buyer', 'buyer-name']` becomes `[{'buyer': [None, {'name': [None]}]}]`. A single path such as `['a-b-c']` compresses to `[{'a-b-c': [None]}]`. Empty input produces `[]`. Duplicate names are removed; spelling and case are preserved. Maps use Python's case-sensitive alphabetical ordering by key, with the terminal `None` before child maps.

This is a lexical hierarchy of TED search-field names; shared words do not establish formal ontology relationships. The spelling `onthology_mapper` follows the requested filename.

In [25]:
import csv
from pathlib import Path
from pprint import pprint

# Locate the input from the repository root, TED folder, or a descendant folder.
def find_search_fields(start):
    for folder in (start, *start.parents):
        for candidate in (
            folder / "List_of_search_fields.csv",
            folder / "sources" / "TED" / "List_of_search_fields.csv",
        ):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Cannot locate sources/TED/List_of_search_fields.csv")


input_path = find_search_fields(Path.cwd())
with input_path.open(encoding="utf-8-sig", newline="") as handle:
    rows = csv.reader(handle)
    header = next(rows)
    search_fields = [row[0].strip() for row in rows if row and row[0].strip()]

print(f"Column: {header[0]}; rows: {len(search_fields)}; distinct: {len(set(search_fields))}")

Column: Search field; rows: 1340; distinct: 1340


In [26]:
# Step 0: group each first word with its distinct following suffixes.
def first_level_map(names):
    names = list(dict.fromkeys(names))
    if names == [None]:
        return names

    groups = {}
    for name in names:
        if name is None:
            key, suffix = None, None
        else:
            key, separator, remainder = name.partition("-")
            suffix = remainder if separator and remainder else None
        groups.setdefault(key, {})[suffix] = None
    return {key: list(following) for key, following in groups.items()}


# Step 1: compress shared leading words without crossing a terminal suffix.
def compress_group(key, suffixes):
    while suffixes and None not in suffixes:
        parts = [suffix.partition("-") for suffix in suffixes]
        word = parts[0][0]
        if any(part[0] != word for part in parts):
            break
        key = f"{key}-{word}"
        suffixes = list(dict.fromkeys(
            remainder if separator and remainder else None
            for _, separator, remainder in parts
        ))
    return key, suffixes


# Step 2: recursively turn each compressed suffix group into the next level.
def hierarchical_map(names):
    groups = first_level_map(names)
    if groups == [None]:
        return groups

    result = []
    for key, suffixes in groups.items():
        if key is None:
            result.append(None)
            continue
        key, suffixes = compress_group(key, suffixes)
        result.append({key: hierarchical_map(suffixes)})
    # Sort after compression because the final keys determine sibling order.
    result.sort(key=lambda item: (
        item is not None, next(iter(item)) if item is not None else ""
    ))
    return result


def restore_names(node, prefix=()):
    assert isinstance(node, list), "Every level must be a list"
    assert all(item is None or (isinstance(item, dict) and len(item) == 1) for item in node)
    keys = [next(iter(item)) if item is not None else None for item in node]
    assert all(next(iter(item)) is not None for item in node if item is not None)
    assert len(keys) == len(set(keys)), "Sibling keys and terminal markers must be unique"
    assert keys == sorted(keys, key=lambda key: (key is not None, key or ""))
    names = []
    for item in node:
        if item is None:
            names.append("-".join(prefix))
            continue
        key, child = next(iter(item.items()))
        names.extend(restore_names(child, (*prefix, key)))
    return names

## Check compression and terminal names

These examples cover a shared multiword prefix, a name that is also a parent, duplicates, an empty list, and the `[None]` stopping rule.

In [27]:
assert hierarchical_map([None]) == [None]
assert hierarchical_map([]) == []
assert hierarchical_map(["a-b-d", "a-b-c", "a-b-c"]) == [
    {"a-b": [{"c": [None]}, {"d": [None]}]}
]
assert hierarchical_map(["z-q-r", "a-b-c", "a-b", "a"]) == [
    {"a": [None, {"b": [None, {"c": [None]}]}]},
    {"z-q-r": [None]},
]
assert hierarchical_map(["a-bc", "a-b"]) == [
    {"a": [{"b": [None]}, {"bc": [None]}]}
]
assert hierarchical_map(["a-z", "a!-x"]) == [
    {"a!-x": [None]}, {"a-z": [None]}
]
assert hierarchical_map(["alfa", "alfa-beta", "alfa-beta-gamma"]) == [
    {"alfa": [None, {"beta": [None, {"gamma": [None]}]}]}
]
print("Example checks passed.")

Example checks passed.


In [28]:
# Build the full hierarchy and verify that it preserves every distinct name.
mapping = hierarchical_map(search_fields)

In [24]:
restored = restore_names(mapping)
assert len(restored) == len(set(search_fields))
assert set(restored) == set(search_fields)
print(f"Verified {len(restored)} distinct fields; {len(mapping)} top-level groups.")

Verified 1340 distinct fields; 143 top-level groups.


In [29]:
# Display the complete hierarchy with alphabetically ordered sibling maps.
pprint(mapping, sort_dicts=False, width=100)

[{'AC': [None]},
 {'FT': [None]},
 {'HA': [None]},
 {'NL': [None]},
 {'RN': [None]},
 {'RP': [None]},
 {'TD': [None]},
 {'TY': [None]},
 {'accessibility': [{'justification-lot': [None]}, {'lot': [None]}]},
 {'acquiring-cpb-buyer': [None]},
 {'activity-sector': [None]},
 {'additional': [{'classification': [{'lot': [None]},
                                     {'part': [None]},
                                     {'proc': [None]},
                                     {'type': [{'lot': [None]},
                                               {'part': [None]},
                                               {'proc': [None]}]}]},
                 {'info': [{'glo': [None]}, {'part': [None]}, {'proc': [None]}]},
                 {'information': [None, {'lot': [None]}]}]},
 {'announcement': [{'title': [None]}, {'url': [None]}]},
 {'assets-related-contract-extension-indicator-lot': [None]},
 {'authority-main-activity': [None]},
 {'award': [{'criteria': [{'complicated-glo': [None]}, {'order-justi

In [33]:
k_list = [(list(x.keys())[0], len(restore_names(list(x.values())[0]))) for x in mapping]
k_list.sort(key=lambda item: item[1], reverse=True)
k_list

[('organisation', 420),
 ('touchpoint', 364),
 ('buyer', 32),
 ('place', 28),
 ('winner', 27),
 ('contract', 24),
 ('review', 23),
 ('framework', 18),
 ('award', 17),
 ('tender', 17),
 ('deadline', 14),
 ('business', 12),
 ('document', 12),
 ('subcontracting', 12),
 ('additional', 11),
 ('selection', 9),
 ('ubo', 9),
 ('electronic', 8),
 ('estimated-value', 8),
 ('change', 7),
 ('eu', 7),
 ('main', 7),
 ('notice', 7),
 ('result', 7),
 ('dispatch', 6),
 ('legal-basis', 6),
 ('modification', 6),
 ('previous', 6),
 ('url', 6),
 ('concession', 5),
 ('duration', 5),
 ('eed', 5),
 ('group-framework', 5),
 ('procedure', 5),
 ('reserved', 5),
 ('description', 4),
 ('energy', 4),
 ('internal-identifier', 4),
 ('procurement', 4),
 ('public-opening', 4),
 ('registration', 4),
 ('submission', 4),
 ('title', 4),
 ('tool', 4),
 ('direct-award-justification', 3),
 ('dps', 3),
 ('environmental', 3),
 ('exclusion-grounds', 3),
 ('identifier', 3),
 ('lots', 3),
 ('non', 3),
 ('publication', 3),
 ('sme',